# Building Segmentation in Satellite Images
## Comparison Experiment — BCE + Dice Loss

Project ID: 22 · Elena Tuturina · Person 2

**Question:** Does adding an overlap-based Dice term to BCE improve building IoU/Dice
for the existing ImageNet-pretrained U-Net + ResNet18?

This notebook is derived from `04_baseline.ipynb` at commit `2c418ba`.
The only major experimental change is the loss:

`L = BCEWithLogitsLoss(logits, y) + mean_n[1 - (2 sum(p_n y_n) + 1) / (sum(p_n) + sum(y_n) + 1)]`,
where `p = sigmoid(logits)`. Dice is soft and computed per patch, including empty
masks, then averaged over the batch. Both terms have coefficient 1; smoothing is 1.

The expected benefit is greater attention to building overlap despite foreground /
background imbalance. Improvement is a hypothesis, not a guaranteed result.

**Fixed:** original split (4932/144/360 patches), preprocessing, U-Net + ResNet18,
ImageNet encoder weights, batch size 8, Adam (lr 0.001), 10 epochs, seed 42,
full precision, no augmentation, threshold 0.5, lowest validation-loss checkpoint.
Here validation loss means the new combined objective. Its numerical value must
not be compared with the baseline BCE loss.

Run the existing `02_prepare_subset.ipynb` first if patches are absent. Then select
the project `.venv` kernel, **Restart Kernel and Run All Cells**, and save this
notebook with outputs. Do not rerun only the training cell: that continues from
the current model/RNG state. A complete rerun replaces this experiment's artifacts.

Threshold sensitivity, detailed failure analysis and the team's final report
belong to Persons 3 and 4 and are not performed here.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU required for this run. Select the project .venv kernel and check nvidia-smi.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/preprocessing.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside CV-P22-building-segmentation.")
print("Project root:", PROJECT_ROOT)

In [ ]:
import os
import sys
import time
import subprocess
from datetime import datetime, timezone

# Keep downloaded pretrained weights local to this checkout.
os.environ.setdefault("TORCH_HOME", str(PROJECT_ROOT / ".cache" / "torch"))
os.environ.setdefault("HF_HOME", str(PROJECT_ROOT / ".cache" / "huggingface"))
import segmentation_models_pytorch as smp

sys.path.insert(0, str(PROJECT_ROOT))
from src.preprocessing import load_and_preprocess
from src.losses import BCEDiceLoss
from src.comparison_support import validate_comparison_data, runtime_environment, write_json, sha256

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
ENVIRONMENT = runtime_environment(PROJECT_ROOT)
print("Runtime:", ENVIRONMENT["packages"])

In [ ]:
import pandas as pd

SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

train_df = pd.read_csv(SPLITS_DIR / "train.csv")
val_df = pd.read_csv(SPLITS_DIR / "val.csv")
test_df = pd.read_csv(SPLITS_DIR / "test.csv")

print(len(train_df))
print(len(val_df))
print(len(test_df))

# Read-only preflight; reuse the existing split files and processed data.
DATA_SUMMARY = validate_comparison_data(
    PROJECT_ROOT, {"train": train_df, "val": val_df, "test": test_df}
)
print("All baseline image-mask paths and source-level splits verified.")

In [ ]:
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("Random seed:", SEED)

In [ ]:
from torch.utils.data import Dataset, DataLoader


class BuildingDataset(Dataset):
    def __init__(self, dataframe, project_root):
        self.dataframe = dataframe.reset_index(drop=True)
        self.project_root = project_root

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = self.project_root / row["image_path"]
        mask_path = self.project_root / row["mask_path"]

        image, mask = load_and_preprocess(
            image_path,
            mask_path
        )

        return image, mask

In [ ]:
train_dataset = BuildingDataset(train_df, PROJECT_ROOT)
val_dataset = BuildingDataset(val_df, PROJECT_ROOT)
test_dataset = BuildingDataset(test_df, PROJECT_ROOT)

In [ ]:
BATCH_SIZE = 8

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    generator=generator

)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

### Preserve baseline initialization and loader order

The baseline consumes one training batch before model construction, then a second
batch for a shape-check forward in training mode. These steps are deliberately
retained in their original order: removing them changes the shuffled batch order
and BatchNorm state. Run the notebook from the beginning once per experiment.

In [ ]:
images, masks = next(iter(train_loader))

print(images.shape)
print(masks.shape)

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

In [ ]:
model = smp.Unet(
    encoder_name="resnet18",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1
)

model = model.to(device)

In [ ]:
images, masks = next(iter(train_loader))

images = images.to(device)
masks = masks.to(device)

with torch.no_grad():
    outputs = model(images)

print("Images:", images.shape)
print("Masks:", masks.shape)
print("Outputs:", outputs.shape)

In [ ]:
from tqdm import tqdm
import pandas as pd
import torch


# Loss and optimizer
criterion = BCEDiceLoss(smooth=1.0)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


def compute_segmentation_stats(logits, targets, threshold=0.5):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    intersection = (predictions * targets).sum().item()

    prediction_sum = predictions.sum().item()
    target_sum = targets.sum().item()

    union = prediction_sum + target_sum - intersection

    return intersection, union, prediction_sum, target_sum


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    total_intersection = 0.0
    total_union = 0.0
    total_prediction_sum = 0.0
    total_target_sum = 0.0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for images, masks in progress_bar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, masks)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * images.size(0)

        intersection, union, pred_sum, target_sum = (
            compute_segmentation_stats(
                outputs.detach(),
                masks
            )
        )

        total_intersection += intersection
        total_union += union
        total_prediction_sum += pred_sum
        total_target_sum += target_sum

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    average_loss = total_loss / len(loader.dataset)

    iou = (
        total_intersection
        / (total_union + 1e-7)
    )

    dice = (
        2 * total_intersection
        / (
            total_prediction_sum
            + total_target_sum
            + 1e-7
        )
    )

    return average_loss, iou, dice


def evaluate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0

    total_intersection = 0.0
    total_union = 0.0
    total_prediction_sum = 0.0
    total_target_sum = 0.0

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for images, masks in progress_bar:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, masks)

            total_loss += loss.item() * images.size(0)

            intersection, union, pred_sum, target_sum = (
                compute_segmentation_stats(
                    outputs,
                    masks
                )
            )

            total_intersection += intersection
            total_union += union
            total_prediction_sum += pred_sum
            total_target_sum += target_sum

    average_loss = total_loss / len(loader.dataset)

    iou = (
        total_intersection
        / (total_union + 1e-7)
    )

    dice = (
        2 * total_intersection
        / (
            total_prediction_sum
            + total_target_sum
            + 1e-7
        )
    )

    return average_loss, iou, dice


# Training settings
NUM_EPOCHS = 10

history = []

best_val_loss = float("inf")

MODEL_PATH = (
    PROJECT_ROOT
    / "results"
    / "comparison_unet_resnet18.pt"
)

METRICS_DIR = (
    PROJECT_ROOT
    / "results"
    / "metrics"
)

MODEL_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def synchronize():
    torch.cuda.synchronize()


synchronize()
training_started = time.perf_counter()

# Training loop
for epoch in range(1, NUM_EPOCHS + 1):

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")

    synchronize()
    epoch_started = time.perf_counter()
    train_loss, train_iou, train_dice = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    val_loss, val_iou, val_dice = evaluate(
        model,
        val_loader,
        criterion,
        device
    )

    synchronize()
    epoch_seconds = time.perf_counter() - epoch_started
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_iou": train_iou,
        "train_dice": train_dice,
        "val_loss": val_loss,
        "val_iou": val_iou,
        "val_dice": val_dice,
        "epoch_seconds": epoch_seconds,
    })

    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train IoU: {train_iou:.4f} | "
        f"Train Dice: {train_dice:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val IoU:   {val_iou:.4f} | "
        f"Val Dice:  {val_dice:.4f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

        torch.save(
            model.state_dict(),
            MODEL_PATH
        )

        print("Best checkpoint saved.")

    # Preserve completed epochs even if a later epoch is interrupted.
    pd.DataFrame(history).to_csv(METRICS_DIR / "comparison_training_history.csv", index=False)


synchronize()
training_seconds = time.perf_counter() - training_started
print(f"Training + epoch validation + checkpoint writes: {training_seconds:.1f} s")

# Save training history
history_df = pd.DataFrame(history)

history_df.to_csv(
    METRICS_DIR / "comparison_training_history.csv",
    index=False
)

print("\nTraining finished.")
print("Best validation loss:", round(best_val_loss, 4))
print("Best model saved to:", MODEL_PATH)

history_df

In [ ]:
model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)
model.eval()

print("Best checkpoint loaded.")

In [ ]:
# The checkpoint is frozen before accessing the test split.
synchronize()
test_started = time.perf_counter()
test_loss, test_iou, test_dice = evaluate(model, test_loader, criterion, device)
synchronize()
test_evaluation_seconds = time.perf_counter() - test_started

print(f"Test combined loss: {test_loss:.4f}")
print(f"Test IoU:  {test_iou:.4f}")
print(f"Test Dice: {test_dice:.4f}")
print(f"Test evaluation including data loading and metrics: {test_evaluation_seconds:.2f} s")

In [ ]:
best_epoch_row = history_df.loc[
    history_df["val_loss"].idxmin()
]

comparison_metrics = pd.DataFrame([
    {
        "model": "U-Net + ResNet18",
        "encoder_weights": "ImageNet",
        "loss": "BCEWithLogitsLoss + soft Dice",
        "dice_smooth": 1.0,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "training_seconds": training_seconds,
        "test_evaluation_seconds": test_evaluation_seconds,
        "optimizer": "Adam",
        "learning_rate": 1e-3,
        "threshold": 0.5,
        "best_epoch": int(best_epoch_row["epoch"]),
        "best_val_loss": best_epoch_row["val_loss"],
        "best_val_iou": best_epoch_row["val_iou"],
        "best_val_dice": best_epoch_row["val_dice"],
        "test_loss": test_loss,
        "test_iou": test_iou,
        "test_dice": test_dice,
    }
])

comparison_metrics.to_csv(
    METRICS_DIR / "comparison_metrics.csv",
    index=False
)

comparison_metrics

In [ ]:
images, masks = next(iter(test_loader))

images_device = images.to(device)

with torch.no_grad():
    logits = model(images_device)

probabilities = torch.sigmoid(logits)
predictions = (probabilities >= 0.5).float().cpu()

print("Images:", images.shape)
print("Ground truth:", masks.shape)
print("Predictions:", predictions.shape)

In [ ]:
IMAGENET_MEAN_TENSOR = torch.tensor(
    [0.485, 0.456, 0.406]
).view(3, 1, 1)

IMAGENET_STD_TENSOR = torch.tensor(
    [0.229, 0.224, 0.225]
).view(3, 1, 1)


def denormalize_image(image):
    image = image.cpu()

    image = (
        image * IMAGENET_STD_TENSOR
        + IMAGENET_MEAN_TENSOR
    )

    return image.clamp(0, 1)

In [ ]:
import matplotlib.pyplot as plt

NUM_EXAMPLES = 4

fig, axes = plt.subplots(
    NUM_EXAMPLES,
    3,
    figsize=(12, 4 * NUM_EXAMPLES)
)

for i in range(NUM_EXAMPLES):
    image = denormalize_image(images[i])
    image = image.permute(1, 2, 0).numpy()

    ground_truth = masks[i, 0].numpy()
    prediction = predictions[i, 0].numpy()

    axes[i, 0].imshow(image)
    axes[i, 0].set_title("Aerial image")

    axes[i, 1].imshow(ground_truth, cmap="gray")
    axes[i, 1].set_title("Ground truth")

    axes[i, 2].imshow(prediction, cmap="gray")
    axes[i, 2].set_title("BCE + Dice prediction")

    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
fig.savefig(PROJECT_ROOT / "figures" / "comparison_examples.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from PIL import Image
import numpy as np

PREDICTIONS_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
)

PREDICTIONS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

for i in range(min(10, len(predictions))):
    prediction = predictions[i, 0].numpy()

    prediction_image = (
        prediction * 255
    ).astype(np.uint8)

    Image.fromarray(prediction_image).save(
        PREDICTIONS_DIR
        / f"comparison_prediction_{i:02d}.png"
    )

print("Predictions saved.")

# Fixed first eight CSV rows, matching the baseline's saved examples.
selected = test_df.iloc[:len(predictions)].copy()
selected["prediction_path"] = [
    f"results/predictions/comparison_prediction_{i:02d}.png" for i in range(len(selected))
]
selected.to_csv(METRICS_DIR / "comparison_selected_predictions.csv", index=False)

### Initial quantitative comparison

The following small comparison is for the second-approach milestone. The baseline
numbers are the existing team's saved results, not a new baseline run. Metrics use
global pixel totals (micro IoU/Dice), including padded and background-only pixels,
exactly as in the baseline. Differences are from a single run per condition;
hardware and any differences in software versions must be considered when interpreting them.

In [ ]:
baseline_metrics = pd.read_csv(METRICS_DIR / "baseline_metrics.csv")
comparison_rows = []
for split, prefix in [("validation", "best_val"), ("test", "test")]:
    for metric in ["iou", "dice"]:
        column = f"{prefix}_{metric}"
        reference = float(baseline_metrics.iloc[0][column])
        current = float(comparison_metrics.iloc[0][column])
        comparison_rows.append({
            "split": split, "metric": metric,
            "baseline": reference, "bce_dice": current,
            "delta_percentage_points": 100 * (current - reference),
        })
initial_comparison = pd.DataFrame(comparison_rows)
initial_comparison.to_csv(METRICS_DIR / "comparison_initial_comparison.csv", index=False)
display(initial_comparison)
print("Checkpoint selected using validation only; test threshold fixed at 0.5.")


### Handoff artifacts and runtime

Export probabilities for all validation/test patches so Person 3 can perform their
own threshold and failure analysis. These caches and the checkpoint stay local
because of their size. No thresholds are tuned here. Sample IDs are stored in CSV order.

In [ ]:
CACHE_DIR = PROJECT_ROOT / "results" / "comparison_probabilities"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
export_started = time.perf_counter()
model.eval()
for split, loader, frame in [("val", val_loader, val_df), ("test", test_loader, test_df)]:
    chunks = []
    with torch.no_grad():
        for batch_images, _ in tqdm(loader, desc=f"Export {split}"):
            batch_logits = model(batch_images.to(device, non_blocking=True))
            chunks.append(torch.sigmoid(batch_logits).cpu().numpy())
    np.savez_compressed(
        CACHE_DIR / f"{split}.npz",
        probabilities=np.concatenate(chunks, axis=0),
        image_paths=frame.image_path.to_numpy(dtype=str),
        mask_paths=frame.mask_path.to_numpy(dtype=str),
        source_images=frame.source_image.to_numpy(dtype=str),
        patch_indices=frame.patch_idx.to_numpy(),
    )
synchronize()
export_seconds = time.perf_counter() - export_started

manifest = {
    "status": "completed", "started_utc": RUN_STARTED_UTC,
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "baseline_commit": "2c418ba", "contributor": "Elena Tuturina",
    "research_question": "Does BCE + soft Dice improve building IoU/Dice over BCE?",
    "changed_factor": "loss", "loss": "BCEWithLogitsLoss + mean per-patch soft Dice loss",
    "loss_coefficients": [1, 1], "dice_smooth": 1.0,
    "architecture": "Unet", "encoder": "resnet18", "encoder_weights": "imagenet",
    "seed": SEED, "batch_size": BATCH_SIZE, "epochs": NUM_EPOCHS,
    "optimizer": "Adam", "learning_rate": 0.001, "threshold": 0.5,
    "checkpoint_selection": "minimum validation combined loss",
    "metric_aggregation": "global pixel totals; epsilon=1e-7",
    "data": DATA_SUMMARY, "environment": ENVIRONMENT,
    "checkpoint_path": str(MODEL_PATH.relative_to(PROJECT_ROOT)),
    "checkpoint_sha256": sha256(MODEL_PATH),
    "best_epoch": int(best_epoch_row["epoch"]),
    "training_seconds_including_epoch_validation": training_seconds,
    "test_evaluation_seconds_including_loading_and_metrics": test_evaluation_seconds,
    "test_evaluation_seconds_per_patch": test_evaluation_seconds / len(test_dataset),
    "probability_export_seconds_including_compression": export_seconds,
    "baseline_reference_sha256": sha256(METRICS_DIR / "baseline_metrics.csv"),
}
write_json(METRICS_DIR / "comparison_run.json", manifest)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(METRICS_DIR / "comparison_environment.txt").write_text(freeze.stdout, encoding="utf-8")
print("Saved metrics, selected predictions, probabilities, checkpoint, environment and timing.")
print("Save this notebook with outputs. The milestone report can now use the actual results.")
